# Error taxonomy

Goal: find which kinds of financial language the frozen FinBERT CNN gets wrong, and check which of these errors the bigger FinBERT transformer fixes.

1. **Error pool**: training sentences the CNN misclassifies, using its out-of-fold predictions from notebook 02 (`results/hard_samples/band_train.csv`), so every prediction comes from a model that never saw the sentence.
2. **Coding sheets**: a sample of the pool, coded by hand with the categories in the codebook below. 20 shared sentences measure agreement between coders.
3. **Agreement** (Cohen's kappa) and the **taxonomy table**: per category, how many sentences, how many all-agree, and how many the transformer gets right (report section 5.5).
4. **Numbers and figures for the report** (sections 5.4 and 5.5).

Only the training set is used to build the taxonomy; val and test stay untouched.

In [1]:
import os
import sys

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split

sys.path.append("..")

pd.set_option("display.max_colwidth", None)
os.makedirs("../results/taxonomy", exist_ok=True)

## 1. Error pool
CNN out-of-fold predictions, hard mask and all-agree flag all come from notebook 02 (`results/hard_samples/band_train.csv`). `is_hard` and `all_agree` are kept in the pool for the analysis, but left off the coding sheets so they don't steer the coders.

In [2]:
train = pd.read_csv("../data/train.csv")
cnn = pd.read_csv("../results/hard_samples/band_train.csv")
assert (cnn.sentence.values == train.sentence.values).all()

errors = cnn[["sentence", "label", "pred", "max_prob", "is_hard", "all_agree"]].copy()
errors.insert(0, "train_idx", np.arange(len(train)))
errors = errors.rename(columns={"pred": "cnn_pred", "max_prob": "cnn_max_prob"})
errors["cnn_max_prob"] = errors.cnn_max_prob.round(3)
errors["n_words"] = train.sentence.str.split().str.len()

pool = errors[errors.cnn_pred != errors.label].copy()
pool["error_type"] = pool.label + " -> " + pool.cnn_pred
pool.to_csv("../results/taxonomy/error_pool.csv", index=False)

print(f"CNN wrong (pool): {len(pool)} / {len(train)}, of which hard: {pool.is_hard.sum()}, all-agree: {pool.all_agree.sum()}")
pool.error_type.value_counts()

CNN wrong (pool): 615 / 2901, of which hard: 193, all-agree: 114


error_type
positive -> neutral     273
neutral -> positive     123
negative -> neutral     100
negative -> positive     52
neutral -> negative      45
positive -> negative     22
Name: count, dtype: int64

## 2. Codebook

A sentence can get **several** categories. Mark a category with `1`.

| Column | Meaning | Example sentence |
|---|---|---|
| `ambiguous_label` | the sentence could reasonably get another label | "Sanoma will continue to focus on investing in digital media"; "However, the total orders received will still be above last year's levels" |
| `domain_required` | the sentiment needs domain knowledge to see | "Technopolis is Europe's leading operator of technoparks" |
| `mixed_sentiment` | positive and negative information in one sentence | "Profit up but sales down" |
| `corporate_action` | stock acquisition, company-internal events, collaboration | "Ponsse has agreed with Volvo on the start of cooperation in Latin America" |
| `numerical_comparison` | the sentiment depends on comparing numerical figures | "EPS stood at EUR0.07 versus EUR0.11" |
| `other` | none of the above: describe it in `notes` (candidates for new categories) | |

Rules:
- Code what makes the sentence **hard**, not every phenomenon it contains.
- `ambiguous_label` = label noise / disagreement. These are excluded from targeted training later, so use it honestly.
- Do not change the gold label; use `notes` if you think it is wrong.

In [3]:
CATEGORIES = ["ambiguous_label", "domain_required", "mixed_sentiment", "corporate_action",
              "numerical_comparison", "other"]
CODERS = ["A", "B", "C", "D"]
N_UNIQUE, N_SHARED = 50, 20   # per coder: 50 own sentences + 20 shared by everyone (for agreement)

# Stratified sample over error types, so the coded sentences have the same mix of mistakes as the pool.
sample, _ = train_test_split(pool, train_size=N_SHARED + N_UNIQUE * len(CODERS),
                             stratify=pool.error_type, random_state=42)
sample = sample.sample(frac=1, random_state=42)
shared = sample.iloc[:N_SHARED]
unique = sample.iloc[N_SHARED:]

columns = ["train_idx", "sentence", "label", "cnn_pred"]
for i, coder in enumerate(CODERS):
    path = f"../results/taxonomy/coding_sheet_{coder}.csv"
    if os.path.exists(path):   # never overwrite a sheet someone may already be coding
        print(f"{path} exists, not overwritten")
        continue
    sheet = pd.concat([shared, unique.iloc[i * N_UNIQUE:(i + 1) * N_UNIQUE]])[columns]
    sheet = sheet.sample(frac=1, random_state=i)          # shared rows mixed in, not all at the top
    sheet["shared"] = sheet.train_idx.isin(shared.train_idx)
    for category in CATEGORIES:
        sheet[category] = ""
    sheet["notes"] = ""
    sheet.to_csv(path, index=False)

print(f"{len(CODERS)} sheets x {N_UNIQUE + N_SHARED} sentences ({N_SHARED} shared), "
      f"{len(sample)} distinct sentences out of {len(pool)} in the pool")
sample.error_type.value_counts()

4 sheets x 70 sentences (20 shared), 220 distinct sentences out of 615 in the pool


error_type
positive -> neutral     98
neutral -> positive     44
negative -> neutral     36
negative -> positive    18
neutral -> negative     16
positive -> negative     8
Name: count, dtype: int64

## 3. Load the coded sheets

Sheet A was coded by one team member, sheets B and D by a second one. Sheet C was not coded, so it is not used.
Sheet A calls the clear-sentiment category `explicit_missed`, so we rename it to `clear_sentiment`.

In [ ]:
CATEGORIES = ["ambiguous_label", "domain_required", "corporate_action", "numerical_comparison",
              "clear_sentiment", "mixed_sentiment", "other"]

def load_sheet(path):
    sheet = pd.read_csv(path)
    if "explicit_missed" in sheet.columns:
        sheet = sheet.rename(columns={"explicit_missed": "clear_sentiment"})
    for category in CATEGORIES:
        # empty cell = 0, anything above 0 = the category was ticked
        sheet[category] = pd.to_numeric(sheet[category], errors="coerce").fillna(0) > 0
    return sheet

sheet_a = load_sheet("../results/taxonomy/coding_sheet_A.csv")
sheet_b = load_sheet("../results/taxonomy/coding_sheet_B_labelled.csv")
sheet_d = load_sheet("../results/taxonomy/coding_sheet_D_labelled.csv")
print(len(sheet_a), len(sheet_b), len(sheet_d))

## 4. Agreement between the coders (Cohen's kappa)

Cohen's kappa compares two coders and corrects for the agreement you would get by chance (1 = perfect, 0 = chance level).
We compare coder 1 (sheet A) with coder 2 (sheet B) on the 20 shared sentences. Sheet D gives the same codes as B on these sentences, because the same person coded both.

In [ ]:
shared_ids = sorted(set(sheet_a.train_idx) & set(sheet_b.train_idx) & set(sheet_d.train_idx))
coder1 = sheet_a.set_index("train_idx").loc[shared_ids]
coder2 = sheet_b.set_index("train_idx").loc[shared_ids]
coder2_d = sheet_d.set_index("train_idx").loc[shared_ids]
print("shared sentences:", len(shared_ids))
print("sheet B and D identical on the shared sentences:", (coder2[CATEGORIES] == coder2_d[CATEGORIES]).all().all())

kappas = {}
for category in CATEGORIES:
    if coder1[category].sum() + coder2[category].sum() == 0:
        kappas[category] = np.nan      # nobody ticked it, kappa is not defined
    else:
        kappas[category] = cohen_kappa_score(coder1[category], coder2[category])

# overall: all categories of all shared sentences together
overall = cohen_kappa_score(coder1[CATEGORIES].values.ravel(), coder2[CATEGORIES].values.ravel())
print("overall kappa:", round(overall, 2))
pd.Series(kappas).round(2)

## 5. One row per coded sentence

The 20 shared sentences appear on every sheet, so we keep them once, with the codes of coder 2.

In [ ]:
coded = pd.concat([sheet_a[~sheet_a.train_idx.isin(shared_ids)],
                   sheet_b[~sheet_b.train_idx.isin(shared_ids)],
                   sheet_d[~sheet_d.train_idx.isin(shared_ids)],
                   sheet_b[sheet_b.train_idx.isin(shared_ids)]], ignore_index=True)
print("coded sentences:", len(coded), "| unique:", coded.train_idx.nunique())
print("categories per sentence:", round(coded[CATEGORIES].sum(axis=1).mean(), 2))

## 6. Taxonomy table (report section 5.5)

For every coded sentence we look up:
- `all_agree`: did all the dataset's annotators give the same label (`results/hard_samples/agreement_train.csv`, notebook 07)
- `transformer_right`: does the FinBERT transformer get it right, using its out-of-fold predictions (`results/hard_samples/transformer_band_train.csv`, notebook 05)

In [ ]:
transformer = pd.read_csv("../results/hard_samples/transformer_band_train.csv")
agreement = pd.read_csv("../results/hard_samples/agreement_train.csv")

label_to_id = {"negative": 0, "neutral": 1, "positive": 2}
transformer_pred = transformer[["p_negative", "p_neutral", "p_positive"]].values.argmax(axis=1)
transformer_right = transformer.label.map(label_to_id).values == transformer_pred

coded["all_agree"] = agreement.all_agree.values[coded.train_idx]
coded["transformer_right"] = transformer_right[coded.train_idx]

rows = []
for category in CATEGORIES:
    part = coded[coded[category]]
    rows.append({"category": category, "n": len(part),
                 "% of coded": round(100 * len(part) / len(coded)),
                 "% all-agree": round(100 * part.all_agree.mean()),
                 "fixed by transformer %": round(100 * part.transformer_right.mean())})
rows.append({"category": "all coded", "n": len(coded), "% of coded": 100,
             "% all-agree": round(100 * coded.all_agree.mean()),
             "fixed by transformer %": round(100 * coded.transformer_right.mean())})
taxonomy_table = pd.DataFrame(rows)
taxonomy_table

In [ ]:
print("transformer fixes, all-agree sentences:", round(100 * coded[coded.all_agree].transformer_right.mean()), "%")
print("transformer fixes, disputed sentences: ", round(100 * coded[~coded.all_agree].transformer_right.mean()), "%")
(coded.label + " -> " + coded.cnn_pred).value_counts()

In [ ]:
table = taxonomy_table[taxonomy_table.category != "all coded"].sort_values("fixed by transformer %")
names = [c.replace("_", " ") + f"  (n={n})" for c, n in zip(table.category, table.n)]

plt.figure(figsize=(7, 3.4))
plt.barh(names, table["fixed by transformer %"], color="#2a78d6")
for i, value in enumerate(table["fixed by transformer %"]):
    plt.text(value + 1, i, f"{value}%", va="center", fontsize=9)
plt.axvline(100 * coded.transformer_right.mean(), color="grey", lw=0.8, ls="--")
plt.xlim(0, 100)
plt.xlabel("% of CNN errors the transformer gets right")
plt.title("Which CNN errors a bigger model fixes")
plt.tight_layout()
plt.savefig("../results/figures/taxonomy_transformer_fixes.png", dpi=150)
plt.show()

## 7. Why targeted training does not help (report section 5.4)

All with the transformer's out-of-fold predictions on train. Hard = top probability between 0.4 and 0.6 (strategy 2).

In [ ]:
probs = transformer[["p_negative", "p_neutral", "p_positive"]].values
max_prob = probs.max(axis=1)
in_band = (max_prob >= 0.4) & (max_prob <= 0.6)
wrong = ~transformer_right
disputed = ~agreement.all_agree.values

print("disputed (not all-agree), in the band:", round(100 * disputed[in_band].mean()), "%")
print("disputed, all train sentences:        ", round(100 * disputed.mean()), "%")
print("disputed, among the transformer's errors:", round(100 * disputed[wrong].mean()), "%")
print()
print("transformer errors on train:", wrong.sum())
print("   in the band:", (wrong & in_band).sum())
print("   top probability above 0.9:", (wrong & (max_prob > 0.9)).sum())

The ceiling: F1 on all-agree vs not all-agree sentences, for every targeted training run of notebook 05.

In [ ]:
s1 = pd.read_csv("../results/targeted_transformer/strategy1.csv")
s2 = pd.read_csv("../results/targeted_transformer/strategy2.csv")

names, agree_f1, disputed_f1 = [], [], []
for arm in ["no_weight", "clean_hard", "random", "clean_random"]:
    part = s2[s2.arm == arm]
    names.append("S2 " + arm.replace("_", " "))
    agree_f1.append(part.f1_allagree.mean())
    disputed_f1.append(part.f1_not_allagree.mean())
for p in [10, 20, 30]:
    for arm in ["targeted", "random"]:
        part = s1[(s1.arm == arm) & (s1.percentile == p)]
        names.append(f"S1 {arm} P{p}")
        agree_f1.append(part.f1_allagree.mean())
        disputed_f1.append(part.f1_not_allagree.mean())

print("mean F1 all-agree:", round(np.mean(agree_f1), 3), "| mean F1 not all-agree:", round(np.mean(disputed_f1), 3))

x = np.arange(len(names))
plt.figure(figsize=(9, 3.8))
plt.bar(x - 0.2, agree_f1, 0.4, label="all annotators agreed", color="#2a78d6")
plt.bar(x + 0.2, disputed_f1, 0.4, label="annotators disagreed", color="#eb6834")
plt.xticks(x, names, rotation=35, ha="right", fontsize=8)
plt.ylim(0, 1)
plt.ylabel("macro F1 (val)")
plt.title("Every training condition hits the same two ceilings")
plt.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig("../results/figures/transformer_agreement_ceiling.png", dpi=150)
plt.show()